# SYSTÈME DE TRADING IA — VERSION COMPLÈTE
## LSTM + Transformer + CNN + Agent RL (PPO)

**Ce notebook lance le système COMPLET avec :**
- Téléchargement des données réelles (Yahoo Finance)
- 60+ features techniques, statistiques, microstructure
- Modèle hybride LSTM + Transformer + CNN (1M paramètres)
- Agent Reinforcement Learning PPO
- Backtesting réaliste avec métriques complètes

> **AVANT DE COMMENCER :** Menu Runtime → Change runtime type → T4 GPU

---

## ÉTAPE 1 — Installation des dépendances complètes
⏱️ Durée : 3-5 minutes (une seule fois par session)

In [ ]:
# Installation de toutes les dépendances — VERSION COMPLÈTE avec PyTorch
print("Installation en cours... (3-5 min)")

!pip install -q torch torchvision
!pip install -q numpy pandas scipy scikit-learn
!pip install -q hmmlearn
!pip install -q gymnasium stable-baselines3
!pip install -q loguru tqdm pyyaml joblib
!pip install -q multitasking
!pip install -q yfinance
!pip install -q matplotlib seaborn plotly

import torch
print(f"
PyTorch version  : {torch.__version__}")
print(f"GPU disponible   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
else:
    print("GPU non disponible -> entraînement sur CPU (plus lent)")
print("
Installation terminée !")

## ÉTAPE 2 — Téléchargement du code source
⏱️ Durée : 10 secondes

In [ ]:
import os

# Cloner le dépôt
!git clone -b claude/ai-trading-system-kPgzo https://github.com/openmaxai26-boop/trading.git 2>/dev/null || echo "Déjà cloné"

# Se placer dans le dossier
os.chdir("/content/trading")

# Vérifier la structure
!ls -la
print("
Code source prêt !")

## ÉTAPE 3 — Configuration du système
Modifiez les paramètres selon vos préférences.

In [ ]:
import yaml

# Charger la configuration
with open("config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# ╔═══════════════════════════════════════════╗
# ║  MODIFIEZ CES PARAMÈTRES SELON VOS BESOINS ║
# ╚═══════════════════════════════════════════╝

# Actifs à analyser (ajoutez/supprimez des symboles)
SYMBOLS = ["AAPL", "BTC-USD", "GC=F"]  # Apple, Bitcoin, Or

# Capital de départ
config["portfolio"]["initial_capital"] = 100_000  # 100 000 USD

# Nombre d'époques d'entraînement (plus = meilleur mais plus long)
config["prediction_model"]["training"]["epochs"] = 100
config["prediction_model"]["training"]["patience"] = 15

# Limite de drawdown (stoppe le trading si dépassée)
config["risk"]["max_drawdown_pct"] = 15.0  # -15% maximum

print("Configuration chargée :")
print(f"  Actifs       : {SYMBOLS}")
print(f"  Capital init : {config[chr(39)]portfolio[chr(39)][chr(39)]initial_capital[chr(39)]:,} USD")
print(f"  Epochs max   : {config[chr(39)]prediction_model[chr(39)][chr(39)]training[chr(39)][chr(39)]epochs[chr(39)]}")
print(f"  Max drawdown : -{config[chr(39)]risk[chr(39)][chr(39)]max_drawdown_pct[chr(39)]}%")

## ÉTAPE 4 — Téléchargement des données de marché
⏱️ **Durée : 1-2 minutes**

On télécharge les prix historiques depuis **Yahoo Finance**.
Si le téléchargement échoue (réseau), des données synthétiques sont générées automatiquement.

**Actifs téléchargés :** Actions (AAPL, MSFT, GOOGL, JPM), Crypto (BTC-USD, ETH-USD), Matières premières (GC=F, CL=F)


In [ ]:
import sys
sys.path.insert(0, '/content/trading')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.data.collectors import DataCollector
from src.data.preprocessor import DataPreprocessor

# Charger la config
import yaml
with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Télécharger les données
collector = DataCollector(config)
print('Téléchargement des données de marché...')
raw_data = collector.collect_all()

# Prétraitement
preprocessor = DataPreprocessor(config)
processed_data = preprocessor.process(raw_data, fit=True)

print('')
print('RÉSUMÉ DES DONNÉES TÉLÉCHARGÉES :')
print('-' * 50)
for symbol, df in processed_data.items():
    print(f'  {symbol:10s} : {len(df):5d} jours | {df.index.min().date()} → {df.index.max().date()}')
print('')
print(f'Total : {len(processed_data)} actifs prêts pour l\'entraînement ✓')


## ÉTAPE 5 — Ingénierie des features + Détection de régime
⏱️ **Durée : 1-2 minutes**

On calcule **50+ features** par actif :
- **Techniques** : RSI, MACD, Bollinger Bands, VWAP, ATR, EMA
- **Statistiques** : Volatilité, Sharpe roulant, Skewness, Kurtosis
- **Microstructure** : Order flow, Liquidité, Efficacité du marché

Puis on détecte le **régime de marché** (Bull / Bear / Range / High Volatility)
via un modèle de Markov Caché (HMM).


In [ ]:
from src.features.engineer import FeatureEngineer
from src.models.regime_detector import MarketRegimeDetector

# Ingénierie des features
fe = FeatureEngineer(config)
print('Calcul des features...')
features_data = fe.engineer_all(processed_data)

# Détection de régime
print('')
print('Détection des régimes de marché...')
regime_detector = MarketRegimeDetector(config)

regime_results = {}
for symbol, df in features_data.items():
    try:
        regime_detector.fit(df)
        labels = regime_detector.predict(df)
        regime_results[symbol] = labels
    except Exception as e:
        print(f'  {symbol} : régime non détecté ({e})')

# Afficher un exemple
if regime_results:
    ex_symbol = list(regime_results.keys())[0]
    ex_labels = regime_results[ex_symbol]
    unique, counts = np.unique(ex_labels, return_counts=True)
    REGIME_NAMES = {0: 'BULL', 1: 'BEAR', 2: 'RANGE', 3: 'HIGH_VOL'}
    print('')
    print(f'Régimes pour {ex_symbol} :')
    for u, c in zip(unique, counts):
        pct = 100 * c / len(ex_labels)
        name = REGIME_NAMES.get(int(u), str(u))
        print(f'  {name:10s} : {c:5d} jours ({pct:.1f}%)')

print('')
print('Features et régimes calculés ✓')


## ÉTAPE 6 — Entraînement du modèle LSTM + Transformer + CNN
⏱️ **Durée : 10-30 minutes selon le nombre d'actifs et d'époques (GPU recommandé)**

Le modèle **hybride** combine :
- **LSTM** : capture les dépendances temporelles longues
- **Transformer** : attention multi-têtes sur les patterns
- **CNN** : détecte les patterns locaux (chandeliers)
- **Sortie probabiliste** : prédit P(hausse) avec incertitude

Total : **~1 million de paramètres**


In [ ]:
import torch
from src.models.prediction_model import HybridPredictionModel, ModelTrainer
from src.data.pipeline import DataPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositif utilisé : {device.upper()}')
if device == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')

# Préparer les données
pipeline = DataPipeline(config)

trained_models = {}
training_histories = {}

# Entraîner sur chaque actif (ou seulement les premiers pour aller vite)
MAX_SYMBOLS = config.get('training', {}).get('max_symbols', 3)  # Limiter pour Colab
symbols_to_train = list(features_data.keys())[:MAX_SYMBOLS]

print(f'Entraînement sur {len(symbols_to_train)} actifs : {symbols_to_train}')
print('')

for symbol in symbols_to_train:
    print(f'=== {symbol} ===')
    df = features_data[symbol]
    
    try:
        # Créer les datasets
        train_loader, val_loader, feature_names = pipeline.prepare_loaders(df, symbol)
        n_features = len(feature_names)
        
        # Créer le modèle
        model = HybridPredictionModel(
            input_size=n_features,
            config=config
        ).to(device)
        
        total_params = sum(p.numel() for p in model.parameters())
        print(f'  Paramètres du modèle : {total_params:,}')
        
        # Entraîner
        trainer = ModelTrainer(model, config, device)
        history = trainer.train(train_loader, val_loader)
        
        trained_models[symbol] = model
        training_histories[symbol] = history
        
        # Sauvegarder
        import os
        os.makedirs('models', exist_ok=True)
        torch.save(model.state_dict(), f'models/{symbol}_model.pt')
        print(f'  Modèle sauvegardé : models/{symbol}_model.pt ✓')
        print('')
    except Exception as e:
        print(f'  ERREUR pour {symbol} : {e}')
        import traceback
        traceback.print_exc()
        print('')

print(f'Entraînement terminé : {len(trained_models)} modèles ✓')


## ÉTAPE 7 — Entraînement de l'agent de Renforcement (PPO)
⏱️ **Durée : 5-15 minutes**

L'**agent PPO** (Proximal Policy Optimization) apprend à trader par essais/erreurs :
- **État** : features actuelles + positions + performance
- **Actions** : Acheter (0) / Vendre (1) / Ne rien faire (2)
- **Récompense** : Rendement ajusté au risque (Sharpe ratio)

C'est comme entraîner un joueur d'échecs, mais pour les marchés financiers.


In [ ]:
from src.models.rl_agent import TradingEnvironment, RLAgent

rl_agents = {}
RL_SYMBOLS = list(features_data.keys())[:2]  # 2 actifs pour Colab

print(f'Entraînement RL sur : {RL_SYMBOLS}')
print('')

for symbol in RL_SYMBOLS:
    print(f'=== Agent RL pour {symbol} ===')
    df = features_data[symbol]
    
    try:
        env = TradingEnvironment(df, config)
        agent = RLAgent(env, config)
        
        total_timesteps = config.get('rl', {}).get('total_timesteps', 50000)
        print(f'  Timesteps : {total_timesteps:,}')
        
        agent.train(total_timesteps=total_timesteps)
        
        # Sauvegarder
        agent.save(f'models/{symbol}_rl_agent')
        rl_agents[symbol] = agent
        print(f'  Agent sauvegardé : models/{symbol}_rl_agent ✓')
        print('')
    except Exception as e:
        print(f'  ERREUR : {e}')
        import traceback
        traceback.print_exc()
        print('')

print(f'Agents RL entraînés : {len(rl_agents)} ✓')


## ÉTAPE 8 — Backtesting (Simulation historique)
⏱️ **Durée : 2-5 minutes**

On simule les trades sur l'**historique des 2 dernières années** :
- Méthode : **Walk-Forward Validation** (pas de data leakage)
- Métriques : Rendement total, Sharpe ratio, Max drawdown, Win rate
- Capital initial : 100,000$


In [ ]:
from src.backtesting.backtester import Backtester

backtester = Backtester(config)
backtest_results = {}

print('Backtesting en cours...')
print('')

for symbol in list(trained_models.keys())[:3]:
    print(f'Backtest : {symbol}')
    df = features_data[symbol]
    model = trained_models[symbol]
    
    try:
        result = backtester.run(df, model, symbol=symbol)
        backtest_results[symbol] = result
        
        print(f'  Rendement total  : {result.total_return * 100:+.2f}%')
        print(f'  Sharpe ratio     : {result.sharpe_ratio:.3f}')
        print(f'  Max drawdown     : {result.max_drawdown * 100:.2f}%')
        print(f'  Nombre de trades : {result.n_trades}')
        if result.n_trades > 0:
            print(f'  Win rate         : {result.win_rate * 100:.1f}%')
        print('')
    except Exception as e:
        print(f'  ERREUR : {e}')
        import traceback
        traceback.print_exc()
        print('')

print('Backtesting terminé ✓')


## ÉTAPE 9 — Optimisation du Portefeuille
⏱️ **Durée : 30 secondes**

On calcule l'**allocation optimale** entre les actifs avec 3 méthodes :
1. **Max Sharpe** : Maximise le rendement ajusté au risque
2. **Min Volatilité** : Minimise la volatilité du portefeuille
3. **Risk Parity** : Chaque actif contribue également au risque

C'est la **théorie moderne du portefeuille** de Markowitz (Prix Nobel 1990).


In [ ]:
from src.portfolio.portfolio_manager import PortfolioManager

portfolio_manager = PortfolioManager(config)

# Utiliser les données traitées pour l'optimisation
portfolio_data = {s: features_data[s] for s in list(features_data.keys())[:5]}

print('Optimisation du portefeuille...')
print('')

methods = ['max_sharpe', 'min_volatility', 'risk_parity']
portfolio_results = {}

for method in methods:
    try:
        weights = portfolio_manager.optimize(portfolio_data, method=method)
        portfolio_results[method] = weights
        
        print(f'--- {method.upper().replace("_", " ")} ---')
        for symbol, w in sorted(weights.items(), key=lambda x: -x[1]):
            bar = '#' * int(w * 40)
            print(f'  {symbol:10s} : {w * 100:5.1f}% {bar}')
        print('')
    except Exception as e:
        print(f'  ERREUR {method} : {e}')
        print('')

print('Optimisation du portefeuille terminée ✓')


## ÉTAPE 10 — Gestion du Risque
⏱️ **Durée : 5 secondes**

Le **gestionnaire de risque** surveille en temps réel :
- **VaR** (Value at Risk) : Perte maximale attendue à 95%
- **Drawdown** : Perte depuis le plus haut récent
- **Circuit breakers** : Arrêt automatique si perte > seuil
- **Stop-loss / Take-profit** : Limites par trade


In [ ]:
from src.risk.risk_manager import RiskManager, RiskStatus

risk_manager = RiskManager(config)

print('Test du gestionnaire de risque...')
print('')

# Simuler une série de rendements
np.random.seed(42)
simulated_returns = np.random.normal(0.001, 0.02, 252)  # 1 an de rendements

# Calculer les métriques de risque
var_95 = np.percentile(simulated_returns, 5)
var_99 = np.percentile(simulated_returns, 1)
cumulative = np.cumprod(1 + simulated_returns)
running_max = np.maximum.accumulate(cumulative)
drawdowns = (cumulative - running_max) / running_max
max_dd = drawdowns.min()

print('MÉTRIQUES DE RISQUE (simulation 1 an) :')
print(f'  VaR 95%          : {var_95 * 100:.2f}% par jour')
print(f'  VaR 99%          : {var_99 * 100:.2f}% par jour')
print(f'  Max Drawdown     : {max_dd * 100:.2f}%')
print(f'  Rendement annuel : {(cumulative[-1] - 1) * 100:.2f}%')
print('')

# Tester le circuit breaker
positions = {s: 0.0 for s in list(features_data.keys())[:3]}
portfolio_value = 100000.0

status = risk_manager.check_portfolio_risk(
    positions=positions,
    portfolio_value=portfolio_value,
    daily_return=-0.02
)

print(f'Statut du risque : {status.value}')
print('Gestion du risque opérationnelle ✓')


## ÉTAPE 11 — Visualisations
⏱️ **Durée : 10 secondes**

Graphiques interactifs pour analyser les résultats du système.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 14))
fig.suptitle('SYSTÈME DE TRADING IA — TABLEAU DE BORD', fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Graphe 1: Training loss ──
ax1 = fig.add_subplot(gs[0, 0])
if training_histories:
    sym = list(training_histories.keys())[0]
    h = training_histories[sym]
    ax1.plot(h.get('train_loss', []), label='Train', color='#2196F3')
    ax1.plot(h.get('val_loss', []), label='Val', color='#FF5722')
    ax1.set_title(f'Loss — {sym}', fontsize=10)
    ax1.set_xlabel('Époque')
    ax1.set_ylabel('Loss')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'Aucun modèle entraîné', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Loss', fontsize=10)

# ── Graphe 2: Backtest equity curve ──
ax2 = fig.add_subplot(gs[0, 1:3])
if backtest_results:
    for sym, result in list(backtest_results.items())[:3]:
        if hasattr(result, 'equity_curve') and result.equity_curve is not None and len(result.equity_curve) > 0:
            ax2.plot(result.equity_curve, label=sym)
    ax2.axhline(y=100000, color='gray', linestyle='--', alpha=0.5, label='Capital initial')
    ax2.set_title('Courbes de Capital (Backtest)', fontsize=10)
    ax2.set_xlabel('Jour')
    ax2.set_ylabel('Valeur ($)')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Aucun backtest disponible', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Courbes de Capital', fontsize=10)

# ── Graphe 3: Portfolio allocation (Max Sharpe) ──
ax3 = fig.add_subplot(gs[1, 0])
if 'max_sharpe' in portfolio_results and portfolio_results['max_sharpe']:
    w = portfolio_results['max_sharpe']
    symbols_list = list(w.keys())
    weights_list = [w[s] * 100 for s in symbols_list]
    colors = plt.cm.Set3(np.linspace(0, 1, len(symbols_list)))
    ax3.pie(weights_list, labels=symbols_list, colors=colors, autopct='%1.0f%%', startangle=90)
    ax3.set_title('Allocation Max Sharpe', fontsize=10)
else:
    ax3.text(0.5, 0.5, 'Données indisponibles', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Allocation Max Sharpe', fontsize=10)

# ── Graphe 4: Portfolio allocation (Min Vol) ──
ax4 = fig.add_subplot(gs[1, 1])
if 'min_volatility' in portfolio_results and portfolio_results['min_volatility']:
    w = portfolio_results['min_volatility']
    symbols_list = list(w.keys())
    weights_list = [w[s] * 100 for s in symbols_list]
    colors = plt.cm.Pastel1(np.linspace(0, 1, len(symbols_list)))
    ax4.pie(weights_list, labels=symbols_list, colors=colors, autopct='%1.0f%%', startangle=90)
    ax4.set_title('Allocation Min Volatilité', fontsize=10)
else:
    ax4.text(0.5, 0.5, 'Données indisponibles', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Allocation Min Volatilité', fontsize=10)

# ── Graphe 5: Backtest metrics bar chart ──
ax5 = fig.add_subplot(gs[1, 2])
if backtest_results:
    syms = list(backtest_results.keys())
    returns = [backtest_results[s].total_return * 100 for s in syms]
    bar_colors = ['#4CAF50' if r >= 0 else '#F44336' for r in returns]
    bars = ax5.bar(syms, returns, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax5.axhline(y=0, color='black', linewidth=0.8)
    ax5.set_title('Rendement Total (%)', fontsize=10)
    ax5.set_ylabel('%')
    ax5.tick_params(axis='x', rotation=30)
    ax5.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, returns):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=8)
else:
    ax5.text(0.5, 0.5, 'Aucun backtest disponible', ha='center', va='center', transform=ax5.transAxes)
    ax5.set_title('Rendements', fontsize=10)

# ── Graphe 6: Price + RSI for first symbol ──
ax6 = fig.add_subplot(gs[2, :])
if features_data:
    sym = list(features_data.keys())[0]
    df = features_data[sym].tail(252)  # Dernière année
    ax6.plot(df.index, df['close'], color='#2196F3', linewidth=1, label='Prix clôture')
    ax6.set_title(f'Prix de clôture — {sym} (dernière année)', fontsize=10)
    ax6.set_xlabel('Date')
    ax6.set_ylabel('Prix ($)')
    ax6.legend(fontsize=8)
    ax6.grid(True, alpha=0.3)
    plt.setp(ax6.xaxis.get_majorticklabels(), rotation=30)
else:
    ax6.text(0.5, 0.5, 'Aucune donnée disponible', ha='center', va='center', transform=ax6.transAxes)
    ax6.set_title('Prix', fontsize=10)

plt.savefig('trading_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tableau de bord sauvegardé : trading_dashboard.png ✓')


## RÉSUMÉ FINAL

**Félicitations !** Vous avez entraîné un système de trading IA complet.

### Ce que vous venez de faire :
1. Téléchargé des données réelles de marché
2. Calculé 50+ features techniques, statistiques et de microstructure
3. Détecté les régimes de marché (Bull/Bear/Range/High Vol) via HMM
4. Entraîné un modèle hybride LSTM+Transformer+CNN (~1M paramètres)
5. Entraîné un agent de renforcement PPO
6. Réalisé un backtesting walk-forward
7. Optimisé un portefeuille multi-actifs (Max Sharpe, Min Vol, Risk Parity)

### Fichiers générés :
- `models/*.pt` — Poids des modèles entraînés
- `models/*_rl_agent` — Agents RL sauvegardés
- `trading_dashboard.png` — Tableau de bord visuel

### Pour aller plus loin :
- Augmenter `epochs` dans `config.yaml` pour un meilleur entraînement
- Ajouter plus d'actifs dans la liste
- Ajuster les seuils de risque (`max_drawdown`, `var_limit`)

> **AVERTISSEMENT** : Ce système est à des fins éducatives uniquement.
> Les performances passées ne garantissent pas les performances futures.
> Ne jamais investir plus que ce que vous pouvez vous permettre de perdre.


In [ ]:
print('=' * 60)
print('SYSTÈME DE TRADING IA — RÉSUMÉ')
print('=' * 60)
print('')

col_actif = 'ACTIF'
col_ret = 'RENDEMENT'
col_sharpe = 'SHARPE'
col_dd = 'MAX DD'
col_trades = 'TRADES'
col_win = 'WIN%'

header = f'{col_actif:12s} {col_ret:>12s} {col_sharpe:>8s} {col_dd:>10s} {col_trades:>8s} {col_win:>8s}'
print(header)
print('-' * 62)

if backtest_results:
    for symbol, result in backtest_results.items():
        ret_str = f'{result.total_return * 100:+.2f}%'
        sharpe_str = f'{result.sharpe_ratio:.3f}'
        dd_str = f'{result.max_drawdown * 100:.2f}%'
        trades_str = str(result.n_trades)
        win_str = f'{result.win_rate * 100:.1f}%' if result.n_trades > 0 else 'N/A'
        print(f'{symbol:12s} {ret_str:>12s} {sharpe_str:>8s} {dd_str:>10s} {trades_str:>8s} {win_str:>8s}')
else:
    print('  Aucun résultat de backtest disponible.')
    print('  Entraînez les modèles (Étape 6) avant de lancer le backtest.')

print('')
print('=' * 60)
print('')
print('Modèles sauvegardés :')
import os
if os.path.exists('models'):
    for f in sorted(os.listdir('models')):
        size = os.path.getsize(f'models/{f}')
        print(f'  models/{f}  ({size / 1024:.0f} KB)')
else:
    print('  Aucun modèle sauvegardé encore.')
print('')
print('Système de trading IA opérationnel ✓')
